# HACK AI / Intro to Large Language Modelling

## Use BERT embeddings to analyse policy language used by the Government

### *Mariam Cook*

### *m.cook6@exeter.ac.uk*

### *University of Exeter Centre for Computational Social Science*


## Install transformers

In [ ]:
pip install transformers torch

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

##  Extract Embeddings

*  Generate a BERT embedding of a string of text
*  Calculate the average of these embedding vectors to obtain a single vector representing the text

In [ ]:
'''
last_hidden_states[:, 1:-1, :] # gets all hidden states except the first and last
'''

my_text = 'The quick brown fox jumps over the lazy dog.'

# Tokenize the text
inputs = tokenizer(my_text, return_tensors='pt')

inputs # this should look familiar now, the words in our sentence have been turned into integers mapped to BERT's vocabulary

In [ ]:
# tokens → (model ([Embedding Layer] → [Transformer Encoder Stack x12]) → last_hidden_state

# Obtain contextualised token representations
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
# Remember BERT uses bidirectional attention, so each token attends to all other tokens in both directions.
# Extract the last hidden state from the output of the final transformer layer across all content tokens (excluding CLS and SEP)
# this gives us a matrix of token-level representations
last_hidden_state = outputs.last_hidden_state[:, 1:-1, :]

# Mean pool to get a single embedding vector representing the entire text span.
# Squeeze batch dimension -> shape (embedding_dim,) not (1, embedding_dim)
text_embedding = last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

In [ ]:
# Print the dimensions of the embeddings
print("Shape of the last hidden state (embeddings):", last_hidden_state.shape)

In [ ]:
# check the inputs do indeed map back to the words in the text. The CLS tag is always at the beginning, and SEP marks the end of the text span
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
tokens

In [ ]:
text_embedding

## Load data

Data credit:

*   This data is the result of querying the 2024-25 common crawl UK government subset provided at the 2025 Bristol Datathon. [See here for the full data set (very large).](https://github.com/eshasadia/G5-CommonCrawl/blob/main/news_filtered_2024_p1_1.csv)
*   The data was further restricted to news only by [Meng Lee in our team: Group 5](https://github.com/eshasadia/G5-CommonCrawl/blob/main/data/filtered_2024_p1.csv).
* I restricted the data further to mentions of policy relevant keywords provided by policy experts in Group 5: [see team members here](https://github.com/eshasadia/G5-CommonCrawl/tree/main).

In [ ]:
import pandas as pd

In [ ]:
news_results_df = pd.read_csv('/content/news_results_common_crawl_uk_gov.csv', index_col=0)

news_results_df

In [ ]:
# lets use pandas to remind us which keywords were looked for

news_results_df.result.value_counts()

In [ ]:
# unique source values
news_results_df.source.unique()

## Generate BERT embeddings for the text including these keywords in UK gov press releases

In [ ]:
news_results_df['bert_embedding'] = None # create a new column in the dataframe to store the embeddings

for id, row in news_results_df.iterrows():

  my_text = row.result_text

  # Tokenize the text
  inputs = tokenizer(my_text, return_tensors='pt')

  # Obtain the embeddings
  with torch.no_grad():
      outputs = model(**inputs)

  # Extract the last hidden state (embeddings)
  last_hidden_state = outputs.last_hidden_state[:, 1:-1, :]

  # Print the dimensions of the embeddings
  print("Shape of the last hidden state (embeddings):", last_hidden_state.shape)

  # Define tokens
  #tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

  text_embedding = last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy() # Average contextualised embeddings

  news_results_df.at[id, 'bert_embedding'] = text_embedding


In [ ]:
news_results_df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
news_results_df['bert_embedding'].sample(1).values #check a random sample

In [ ]:
embeddings_array

In [ ]:
print(embeddings_array.shape)

In [ ]:
# Using sklearn's cosine_similarity
embeddings_array = np.array(news_results_df['bert_embedding'].tolist()) # Convert the embedding column to a 2D array
similarity_matrix = cosine_similarity(embeddings_array)

In [ ]:
# If it's 3D (e.g., shape (n, 1, 384)), flatten/squeeze it to 2D
if embeddings_array.ndim == 3:
    embeddings_array = embeddings_array.squeeze()  # Remove singleton dimensions
    # Or if squeeze doesn't work:
    # embeddings_array = embeddings_array.reshape(embeddings_array.shape[0], -1)

print(embeddings_array.shape)  # Should now be (n_samples, embedding_dim)


In [ ]:
similarity_matrix = cosine_similarity(embeddings_array)

In [ ]:
# Convert to dataframe for better readability
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=news_results_df.index,
    columns=news_results_df.index
)

print("Cosine Similarity Matrix:")
print(similarity_df)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# Convert to dataframe (optional but makes labels easier)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=news_results_df.index,
    columns=news_results_df.index
)


In [ ]:
# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    similarity_df,
    cmap='coolwarm',
    center=0.5,
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={'label': 'Cosine Similarity'},
    linewidths=0.5
)
plt.title('Cosine Similarity Matrix Heatmap', fontsize=16)
plt.xlabel('Sample Index')
plt.ylabel('Sample Index')
plt.tight_layout()
plt.show()

# For 72 samples, use numeric indices with larger figure
plt.figure(figsize=(16, 14))
sns.heatmap(
    similarity_df,
    cmap='viridis',
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={'label': 'Cosine Similarity'},
    annot=False,
    xticklabels=True,
    yticklabels=True
)
plt.title('Cosine Similarity Matrix Heatmap (72 samples)', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Add column for year of data into dataframe

news_results_df['year'] = ''
news_results_df['year'] = pd.to_datetime(news_results_df['year'])

for id, row in news_results_df.iterrows():
  if '2024' in row.source:
    news_results_df.at[id, 'year'] = 2024
  else:
    news_results_df.at[id, 'year'] = 2025

In [ ]:
news_results_df.year.values

In [ ]:
news_results_df = news_results_df[['source', 'id', 'year', 'url', 'parent_url', 'cc_url', 'content_truncated',
       'result', 'result_text', 'bert_embedding']]

In [ ]:
news_results_df.columns

In [ ]:
news_results_df

In [ ]:
# save backup dataframe to pickle

import pickle

with open('news_results_df_interim_results.pkl', 'wb') as file:
    pickle.dump(news_results_df, file)

In [ ]:
# import backup file

import pickle

with open('news_results_df_interim_results.pkl', 'rb') as file:
    news_results_df = pickle.load(file)

In [ ]:
import pandas as pd

# check how many mentions of each policy keyword were found in Government news for each year

pd.crosstab(news_results_df['year'], news_results_df['result'])

In [ ]:
policy_embeddings = {}

policy_embeddings[2024] = {}
policy_embeddings[2025] = {}

for id, row in news_results_df.iterrows():
  if row.year == 2024:
    if row.result not in policy_embeddings[2024]:
      policy_embeddings[2024][row.result] = [row.bert_embedding]
    else:
      policy_embeddings[2024][row.result].append(row.bert_embedding)
  else:
    if row.result not in policy_embeddings[2025]:
      policy_embeddings[2025][row.result] = [row.bert_embedding]
    else:
      policy_embeddings[2025][row.result].append(row.bert_embedding)


In [ ]:
policy_embeddings

In [ ]:
import numpy as np
policy_embedding_averages = {}
for key, value in policy_embeddings.items():
  print(key)

  if key not in policy_embedding_averages:
    policy_embedding_averages[key] = {}

  for policy, embeddings in value.items():
    policy_embedding_averages[key][policy] = np.mean(embeddings, axis=0)


In [ ]:
print(len(policy_embedding_averages[2024])) # 10 policy keywords in the 2024 dictionary
print(len(policy_embedding_averages[2025])) # 5 policy keywords in the 2024 dictionary

In [ ]:
policy_embedding_averages

In [ ]:
policy_embedding_averages[2024].keys()

In [ ]:
policy_embedding_averages[2025].keys()

In [ ]:
policy_embedding_averages.keys()

In [ ]:
both_years_policies = ['free school meals', 'School meals', 'Best Start in Life', 'family hubs']
master_comparison_embeddings = {}

for key, value in policy_embedding_averages.items():
  print(key)
  if key not in master_comparison_embeddings:
    master_comparison_embeddings[key] = {}
  print(value.keys())
  for policy, embedding in value.items():
    print(policy)
    if policy in both_years_policies:
      master_comparison_embeddings[key][policy] = embedding


In [ ]:
master_comparison_embeddings[2024].keys() # we are only keeping embeddings for policies spoken about in both years

In [ ]:
# make dataframe from dict

df_compare_embeddings = pd.DataFrame(columns=['policy', 'year', 'embedding'])

for key, value in master_comparison_embeddings.items():
  print(key)
  print(value.keys())
  for policy, embedding in value.items():
    print(policy)
    df_compare_embeddings = pd.concat([df_compare_embeddings, pd.DataFrame([{'policy': policy, 'year': key, 'embedding': embedding}])], ignore_index=True)

In [ ]:
df_compare_embeddings

### Let's check for a drift in language used about policies between 2024 and 2025
Note: in July 2024 the Labour Party won the general election and took over government from the Conservative Party

The cosine similarity bar chart is the better visualisation for comparing how much each topic moved — it measures drift directly in the full 384-dimensional space rather than in a compressed 2D projection. The PCA plot is better for understanding where topics sit relative to each other and the direction of movement, rather than the magnitude.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# ----------  PARAMETERS ----------
year_a = 2024   # set first year
year_b = 2025   # set second year
# -------------------------------------

# 1) Ensure embeddings are numpy arrays
df = df_compare_embeddings.copy()
df['embedding'] = df['embedding'].apply(lambda e: np.array(e, dtype=float))

# 2) If multiple rows per (topic, year), average them
agg = df.groupby(['policy','year'])['embedding'].apply(
    lambda arrs: np.mean(np.vstack(arrs), axis=0)
).reset_index()   # columns: topic, year, embedding

# 3) Build dictionaries for the two years
df_a = agg[agg['year'] == year_a].set_index('policy')
df_b = agg[agg['year'] == year_b].set_index('policy')

topics_a = sorted(df_a.index.tolist())
topics_b = sorted(df_b.index.tolist())

# 4) Per-topic similarity (only topics present in both years)
common_topics = sorted(list(set(topics_a).intersection(topics_b)))

per_topic_sim = []
for topic in common_topics:
    v1 = df_a.loc[topic, 'embedding']
    v2 = df_b.loc[topic, 'embedding']
    # cosine similarity (safe numerical)
    sim = (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12)
    per_topic_sim.append((topic, float(sim)))
per_topic_df = pd.DataFrame(per_topic_sim, columns=['policy','cosine_similarity']).sort_values('cosine_similarity', ascending=False)

# 5) Cross-year topic × topic similarity matrix (rows = topics in year_a, cols = topics in year_b)
mat = np.zeros((len(topics_a), len(topics_b)))
emb_a = np.vstack([df_a.loc[t,'embedding'] for t in topics_a])
emb_b = np.vstack([df_b.loc[t,'embedding'] for t in topics_b])

# use sklearn for stability
sim_matrix = cosine_similarity(emb_a, emb_b)  # shape (len(topics_a), len(topics_b))
cross_df = pd.DataFrame(sim_matrix, index=topics_a, columns=topics_b)


# ---------- OUTPUTS ----------
# per_topic_df : DataFrame with topic and cosine_similarity
# cross_df : DataFrame (topics in year_a x topics in year_b) with pairwise cosines


In [ ]:
mention_counts = pd.crosstab(news_results_df['year'], news_results_df['result'])

# Filter to only common topics
mention_counts = mention_counts[common_topics]
print(mention_counts)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
topics = common_topics
n_topics = len(topics)

# Get mention counts for each year
counts_a = [mention_counts.loc[year_a, t] for t in topics]
counts_b = [mention_counts.loc[year_b, t] for t in topics]
#cos_sims = [per_topic_df.set_index('policy').loc[t, 'cosine_similarity'] for t in topics]

# Replace cosine similarity with semantic drift (1 - cosine_similarity)
cos_sims = [1 - per_topic_df.set_index('policy').loc[t, 'cosine_similarity'] for t in topics]


# Normalise each metric to 0-1 so they're on the same scale
def normalise(lst):
    arr = np.array(lst, dtype=float)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)

counts_a_norm = normalise(counts_a)
counts_b_norm = normalise(counts_b)
cos_sims_norm = normalise(cos_sims)

# Radar setup
#categories = [f'mentions\n{year_a}', f'mentions\n{year_b}', 'cosine\nsimilarity']

# and update the label
categories = [f'mentions\n{year_a}', f'mentions\n{year_b}', 'semantic\ndrift']
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, axes = plt.subplots(1, n_topics, figsize=(4 * n_topics, 4), subplot_kw=dict(polar=True))

for i, (ax, topic) in enumerate(zip(axes, topics)):
    values = [counts_a_norm[i], counts_b_norm[i], cos_sims_norm[i]]
    values += values[:1]  # close the loop

    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
    ax.set_title(topic, fontsize=10, pad=15)

plt.suptitle(f'Topic profiles: mentions & semantic drift {year_a} → {year_b}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
per_topic_df.sort_values('cosine_similarity').style.background_gradient(
    cmap='RdYlGn', subset=['cosine_similarity']
)

In [ ]:
print(per_topic_df.columns)
print(per_topic_df.head())

In [ ]:
print(df_compare_embeddings.columns)
print(df_compare_embeddings.head())

In [ ]:
# Already have per_topic_df with cosine_similarity
per_topic_df.sort_values('cosine_similarity').plot(
    kind='barh', x='policy', y='cosine_similarity', figsize=(8,6)
)
plt.axvline(x=per_topic_df['cosine_similarity'].mean(), color='red', linestyle='--', label='mean')
#plt.title('Topic semantic shift: cosine similarity 2024 → 2025')
plt.title(f'Per-topic cosine similarity: {year_a} → {year_b}')
plt.xlabel('Cosine similarity (lower = more drift) -  (1 = identical)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA with arrows showing movement of common topics

all_topics = list(dict.fromkeys(topics_a + topics_b))

# Fit PCA on all embeddings from both years
emb_a = np.vstack([df_a.loc[t,'embedding'] for t in topics_a])
emb_b = np.vstack([df_b.loc[t,'embedding'] for t in topics_b])
all_emb = np.vstack([emb_a, emb_b])

pca = PCA(n_components=2)
pca.fit(all_emb)

plt.figure(figsize=(10,8))
coords_a = pca.transform(emb_a)
coords_b = pca.transform(emb_b)

plt.scatter(coords_a[:,0], coords_a[:,1], marker='o', label=f'year {year_a}', s=80)
plt.scatter(coords_b[:,0], coords_b[:,1], marker='X', label=f'year {year_b}', s=80)

for topic in common_topics:
    pa = pca.transform(df_a.loc[topic,'embedding'].reshape(1,-1))[0]
    pb = pca.transform(df_b.loc[topic,'embedding'].reshape(1,-1))[0]
    plt.arrow(pa[0], pa[1], (pb[0]-pa[0]), (pb[1]-pa[1]),
              head_width=0.02*np.ptp(coords_a[:,0]), length_includes_head=True, alpha=0.8)
    plt.text(pb[0], pb[1], topic, fontsize=9)

plt.title(f'PCA 2D: topic movement from: {year_a} → {year_b}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
distances = {}
for topic in common_topics:
    pa = pca.transform(df_a.loc[topic,'embedding'].reshape(1,-1))[0]
    pb = pca.transform(df_b.loc[topic,'embedding'].reshape(1,-1))[0]
    distances[topic] = np.linalg.norm(pb - pa)

dist_df = pd.Series(distances).sort_values(ascending=False)
dist_df.plot(kind='bar', figsize=(10,5))

plt.title(f'PCA displacement magnitude per topic: {year_a} → {year_b}')
plt.ylabel('Euclidean distance in PCA space')
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

vmin = cross_df.min().min()
vmax = cross_df.max().max()

plt.figure(figsize=(12,10))
sns.heatmap(
    cross_df,
    cmap='Blues',       # this colour scheme makes it easier to read smaller differences
    vmin=vmin,             # stretch colormap to actual data range
    vmax=vmax,
    annot=len(common_topics) < 15,
    fmt='.2f'
)
plt.title('Cross-year topic cosine similarity')
plt.tight_layout()
plt.show()